# AE

Conditional VAE только с линейными слоями

In [ ]:
import torch.nn as nn
import torch

class CVAE(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, latent_dim, num_classes):
        super(CVAE, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        self.layer_mu = nn.Linear(hidden_dim, latent_dim)
        self.layer.logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + num_classes, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        encoded = self.encoder(x)
        mu = self.layer_mu(encoded)
        logvar = self.layer_logvar(encoded)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        logvar = torch.exp(logvar * 0.5)
        eps = torch.randn_like(logvar)
        return mu + eps * logvar

    def decode(self, latent, y):
        latent = torch.cat([latent, y], dim=1)
        outputs = self.decoder(latent)
        return outputs

    def forward(self, x, y):
        mu, logvar = self.encode(x)
        latent = self.reparameterize(mu, logvar)
        outputs = self.decode(latent, y)
        return outputs

Conditional VAE со свёрточными слоями

In [ ]:
import torch.nn as nn
import torch

class EncoderBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(EncoderBlock, self).__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(input_dim, output_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(output_dim)
        )

    def forward(self, x):
        outputs = self.layers(x)
        return outputs

class DecoderBlock(nn.Module):
    def __init__(self, input_dim, output_dim, last_layer=False):
        super(DecoderBlock, self).__init__()

        self.layers = nn.Sequential(
            nn.ConvTranspose2d(input_dim, output_dim, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid() if last_layer else nn.ReLU()
        )

    def forward(self, x):
        outputs = self.layers(x)
        return outputs

class CVAE(nn.Module):
    def __init__(self, encoder_hidden_dim=16, latent_dim=32, decoder_hidden_dim=16, decoder_proj_channels=4, num_classes=10):
        super(CVAE, self).__init__()
        self.decoder_proj_channels = decoder_proj_channels

        self.encoder = nn.Sequential(
            EncoderBlock(1, encoder_hidden_dim),
            EncoderBlock(encoder_hidden_dim, encoder_hidden_dim),
            nn.Flatten()
        )

        encoder_output_dim = encoder_hidden_dim * 7 * 7
        self.layer_mu = nn.Linear(encoder_output_dim, latent_dim)
        self.layer_logvar = nn.Linear(encoder_output_dim, latent_dim)
    
        decoder_proj_output_dim = decoder_proj_channels * 7 * 7
        self.decoder_proj = nn.Linear(latent_dim + num_classes, decoder_proj_output_dim)

        self.decoder = nn.Sequential(
            DecoderBlock(decoder_proj_channels, decoder_hidden_dim),
            DecoderBlock(decoder_hidden_dim, 1, last_layer=True)
        )

    def encode(self, x):
        encoded = self.encoder(x)
        mu = self.layer_mu(encoded)
        std = self.layer_logvar(encoded)
        return mu, std

    def reparameterize(self, mu, logvar):
        std = torch.exp(logvar * 0.5)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, latent, y):
        latent = torch.cat([latent, y], dim=1)
        latent = self.decoder_proj(latent)
        latent = latent.view(latent.shape[0], self.decoder_proj_channels, 7, 7)
        outputs = self.decoder(latent)
        return outputs

    def forward(self, x, y):
        mu, std = self.encode(x)
        latent = self.reparameterize(mu, std)
        outputs = self.decode(latent, y)
        return outputs, mu, std

def VAELoss(x, y, mu, logvar):
    recon_loss = F.mse_loss(x, y, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + kl_loss
    return total_loss, recon_loss, kl_loss

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F

def evaluate_autoencoder(model, loader, loss_fn, tqdm_leave=True, device=None):
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model.eval()
    losses = []

    for x, y in tqdm(loader, desc='Model evaluation', total=len(loader), leave=tqdm_leave):
        with torch.no_grad():
            x = x.to(device)
            y = F.one_hot(y, num_classes=10).to(torch.float32).to(device)
            y_pred, mu, std = model(x, y)
            loss, reconstruction_loss, kl_loss = loss_fn(y_pred, x, mu, std)
            losses.append(loss.item())
    
    loss = np.mean(losses)
    return loss

def plot_train_results(train_history, val_history):
    plt.plot(train_history, label='Train loss')
    plt.plot(val_history, label='Val loss')
    plt.legend()
    plt.title('Model train results')
    plt.show()

def train_autoencoder(model_params, train_loader, val_loader, lr, epochs, early_stopping_patience=3, plot_results=True):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = CVAE(**model_params).to(device)
    loss_fn = VAELoss
    optimizer = torch.optim.Adam(model.parameters(), lr)
    train_history = []
    val_history = []
    best_val_loss = float('inf')

    for i in tqdm(range(epochs), desc='Model training'):
        train_losses = []
        model.train()

        for x, y in tqdm(train_loader, total=len(train_loader), desc=f'Epoch {i + 1}/{epochs}', leave=False):
            x = x.to(device)
            y = F.one_hot(y, num_classes=10).to(torch.float32).to(device)
            y_pred, mu, std = model(x, y)
            loss, reconstruction_loss, kl_loss = loss_fn(y_pred, x, mu, std)
            train_losses.append(loss.item())
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        
        train_loss = np.mean(train_losses)
        val_loss = evaluate_autoencoder(model, val_loader, loss_fn, tqdm_leave=False)
        print(f'Epoch {i + 1}/{epochs}: Train loss {train_loss:.3f}; Val loss {val_loss:.3f}')

        train_history.append(train_loss)
        val_history.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'model_best_weights.pth')
            early_stopping_counter = 0
            print('New best model saved')
        else:
            early_stopping_counter += 1
            if early_stopping_counter > early_stopping_patience:
                break
    
    best_model = CVAE(**model_params)
    best_weights = torch.load('model_best_weights.pth', weights_only=True)
    best_model.load_state_dict(best_weights)

    if plot_results:
        plot_train_results(train_history, val_history)
    
    return best_model, train_history, val_history

In [ ]:
import math
import matplotlib.pyplot as plt
import random

@torch.no_grad()
def show_grid_vae(model, latent_dim, digit=None, n_reps=16):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval().to(device)

    grid_size = math.ceil(math.sqrt(n_reps))
    noise = torch.randn(n_reps, latent_dim).to(torch.float32).to(device)
    y = F.one_hot(torch.tensor([random.randint(0, 9) for _ in range(n_reps)] if digit is None else [digit] * n_reps),
                  num_classes=10).to(torch.float32).to(device)
    images = model.decode(noise, y).cpu().numpy().reshape(-1, 28, 28)

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(7, 7))
    axes = axes.flatten()

    for i in range(n_reps):
        axes[i].imshow(images[i], cmap='gray')
        axes[i].axis('off')
        
    for i in range(n_reps, len(axes)):
        axes[i].axis('off')

    # plt.tight_layout()
    plt.show()

# GAN

Conditional GAN

In [ ]:
import torch
import torch.nn as nn

class DiscriminatorBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DiscriminatorBlock, self).__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        return self.layers(x)

class Discriminator(nn.Module):
    def __init__(self, embedding_dim=4, conv_hidden_dim=32, classifier_hidden_dim=64, num_classes=10):
        super(Discriminator, self).__init__()

        self.conv = nn.Sequential(
            DiscriminatorBlock(1, conv_hidden_dim),
            DiscriminatorBlock(conv_hidden_dim, conv_hidden_dim),
            nn.Flatten(),
        )
        self.embedding = nn.Embedding(num_embeddings=num_classes, embedding_dim=embedding_dim)
        self.classifier = nn.Sequential(
            nn.Linear(conv_hidden_dim * 7 * 7 + embedding_dim, classifier_hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(classifier_hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x, y):
        conv_outputs = self.conv(x)
        embed = self.embedding(y)
        latent = torch.cat([conv_outputs, embed], dim=1)
        classifier_outputs = self.classifier(latent)
        return classifier_outputs

class GeneratorBlock(nn.Module):
    def __init__(self, in_channels, out_channels, is_last_layer=False):
        super(GeneratorBlock, self).__init__()

        self.layers = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Identity() if is_last_layer else nn.BatchNorm2d(out_channels),
            nn.Sigmoid() if is_last_layer else nn.ReLU()
        )
    
    def forward(self, x):
        return self.layers(x)

class Generator(nn.Module):
    def __init__(self, noise_dim=16, embedding_dim=4, conv_proj_channels=16, generator_hidden_dim=48, num_classes=10):
        super(Generator, self).__init__()

        self.noise_dim = noise_dim
        self.conv_proj_channels = conv_proj_channels

        self.embedding = nn.Embedding(num_embeddings=num_classes, embedding_dim=embedding_dim)
        self.conv_proj = nn.Linear(noise_dim + embedding_dim, conv_proj_channels * 7 * 7)
        self.conv = nn.Sequential(
            GeneratorBlock(conv_proj_channels, generator_hidden_dim),
            GeneratorBlock(generator_hidden_dim, 1, is_last_layer=True)
        )
    
    def forward(self, noise, y):
        embed = self.embedding(y)
        x = torch.cat([noise, embed], dim=1)
        latent = self.conv_proj(x)
        latent = latent.view(latent.shape[0], self.conv_proj_channels, 7, 7)
        outputs = self.conv(latent)
        return outputs

In [ ]:
from tqdm.auto import tqdm
import numpy as np

def train_CGAN(D, G, D_optimizer, G_optimizer, train_loader, epochs):
    device = next(G.parameters()).device
    loss_fn = nn.BCELoss()

    for epoch in range(epochs):
        D.train()
        G.train()       
        D_train_losses = []
        G_train_losses = []

        for x, y in tqdm(train_loader, total=len(train_loader), desc=f'Epoch {epoch + 1}/{epochs}'):
            batch_size = x.size(0)
            x = x.to(device)
            y = y.to(device)

            real_labels = torch.full((batch_size, 1), 0.9, device=device) 
            fake_labels = torch.zeros((batch_size, 1), device=device)

            D_optimizer.zero_grad()

            D_preds_real = D(x, y)
            D_loss_real = loss_fn(D_preds_real, real_labels)

            noise = torch.randn(batch_size, G.noise_dim, device=device)
            fake_images = G(noise, y)
            
            D_preds_fake = D(fake_images.detach(), y)
            D_loss_fake = loss_fn(D_preds_fake, fake_labels)

            D_loss = (D_loss_real + D_loss_fake) / 2
            D_loss.backward()
            D_optimizer.step()

            G_optimizer.zero_grad()

            D_preds_for_G = D(fake_images, y)
            target_ones = torch.ones((batch_size, 1), device=device)
            
            G_loss = loss_fn(D_preds_for_G, target_ones)
            
            G_loss.backward()
            G_optimizer.step()

            D_train_losses.append(D_loss.item())
            G_train_losses.append(G_loss.item())
        
        D_train_loss = np.mean(D_train_losses)
        G_train_loss = np.mean(G_train_losses)
        print(f'Epoch {epoch + 1}/{epochs}: D loss: {D_train_loss:.4f} | G loss: {G_train_loss:.4f}')

In [ ]:
G = Generator(
    noise_dim=16,
    embedding_dim=4,
    conv_proj_channels=16,
    generator_hidden_dim=48,
    num_classes=10
)
D = Discriminator(
    embedding_dim=4,
    conv_hidden_dim=32,
    classifier_hidden_dim=64,
    num_classes=10
)
D_optimizer = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
G_optimizer = torch.optim.Adam(G.parameters(), lr=0.0003, betas=(0.5, 0.999))

train_CGAN(
    D=D,
    G=G,
    D_optimizer=D_optimizer,
    G_optimizer=G_optimizer,
    train_loader=train_loader,
    # val_loader=val_loader,
    epochs=25
)

In [ ]:
import math
import matplotlib.pyplot as plt
import random

@torch.no_grad()
def show_grid_cgan(generator, noise_dim, digit=None, n_reps=16):
    device = next(generator.parameters()).device
    generator.eval()

    grid_size = math.ceil(math.sqrt(n_reps))
    
    noise = torch.randn(n_reps, noise_dim).to(device)
    
    if digit is None:
        y = torch.randint(0, 10, (n_reps,), dtype=torch.long).to(device)
    else:
        y = torch.full((n_reps,), digit, dtype=torch.long).to(device)
        
    images = generator(noise, y).cpu().numpy().reshape(-1, 28, 28)
    y_cpu = y.cpu().numpy()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(7, 7))
    axes = axes.flatten()

    for i in range(n_reps):
        axes[i].imshow(images[i], cmap='gray')
        axes[i].axis('off')
        
        # if digit is None:
            # axes[i].set_title(f"Gen: {y_cpu[i]}")

    for i in range(n_reps, len(axes)):
        axes[i].axis('off')

    # plt.tight_layout()
    plt.show()

show_grid_cgan(G, noise_dim=16, digit=None, n_reps=256) 